# Módulo 9 · Clase 7 — Documentación

**Proyecto Final: Sistema Inteligente de Predicción y Exposición de Modelos**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Docstrings y type hints en Python |
| 2 | OpenAPI/Swagger — documentación automática de la API |
| 3 | README profesional para el proyecto |
| 4 | Git y GitHub — flujo básico |
| 5 | `requirements.txt` y reproducibilidad |

---
## 1. Docstrings y type hints

In [ ]:
# Sin documentar — código funcional pero ilegible para un colaborador
def proc(d, t=7, r=False):
    import numpy as np
    d2 = d.dropna().drop_duplicates()
    d2['so2'] = d2['free sulfur dioxide'] / d2['total sulfur dioxide']
    d2['bv'] = (d2['quality'] >= t).astype(int)
    if r:
        print(d2.shape)
    return d2

print("Función sin documentar: ¿qué hace 'r'? ¿qué devuelve?")

In [ ]:
import pandas as pd
import numpy as np
from typing import Optional

def preprocesar_vinos(
    df: pd.DataFrame,
    umbral_calidad: int = 7,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Limpia y enriquece el dataset de calidad de vinos.

    Pasos que aplica:
    - Elimina filas con valores nulos
    - Elimina duplicados exactos
    - Calcula la feature 'so2_ratio'
    - Crea la variable objetivo binaria 'buen_vino'

    Args:
        df: DataFrame con columnas del dataset Wine Quality (UCI).
        umbral_calidad: Puntuación mínima para clasificar un vino como bueno.
        verbose: Si True, imprime el shape del resultado.

    Returns:
        DataFrame procesado con las columnas originales más 'so2_ratio' y 'buen_vino'.

    Raises:
        KeyError: Si el DataFrame no contiene 'free sulfur dioxide',
                  'total sulfur dioxide' o 'quality'.

    Example:
        >>> df_raw = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
        >>> df_proc = preprocesar_vinos(df_raw, umbral_calidad=7)
        >>> df_proc.columns.tolist()  # incluye 'so2_ratio' y 'buen_vino'
    """
    columnas_req = ['free sulfur dioxide', 'total sulfur dioxide', 'quality']
    for col in columnas_req:
        if col not in df.columns:
            raise KeyError(f"Columna requerida no encontrada: '{col}'")

    df_proc = df.dropna().drop_duplicates().copy()
    df_proc['so2_ratio'] = (
        df_proc['free sulfur dioxide'] / (df_proc['total sulfur dioxide'] + 1e-8)
    )
    df_proc['buen_vino'] = (df_proc['quality'] >= umbral_calidad).astype(int)

    if verbose:
        print(f"Resultado: {df_proc.shape}  (bueno: {df_proc['buen_vino'].mean():.2%})")

    return df_proc

# Ver la docstring
help(preprocesar_vinos)

In [ ]:
# Type hints en funciones más complejas
from pathlib import Path
from typing import Union

def cargar_modelo(
    ruta: Union[str, Path],
    tipo: str = 'sklearn'
) -> object:
    """
    Carga un modelo desde disco.

    Args:
        ruta: Ruta al archivo del modelo (.joblib o .keras).
        tipo: 'sklearn' para joblib, 'keras' para TensorFlow.

    Returns:
        Modelo listo para llamar a .predict().
    """
    import joblib
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"Modelo no encontrado: {ruta}")

    if tipo == 'sklearn':
        return joblib.load(ruta)
    elif tipo == 'keras':
        import tensorflow as tf
        return tf.keras.models.load_model(ruta)
    else:
        raise ValueError(f"Tipo no soportado: '{tipo}'. Usa 'sklearn' o 'keras'.")

# mypy puede verificar tipos en tiempo de desarrollo (no en ejecución)
# mypy api/main.py  →  0 errores si todo está bien tipado
print("✓ Funciones documentadas con docstrings Google-style y type hints")

---
## 2. OpenAPI/Swagger — documentación automática

In [ ]:
# FastAPI genera automáticamente la especificación OpenAPI 3.0
# Disponible en: http://localhost:8000/docs  (Swagger UI)
# y en:          http://localhost:8000/redoc (ReDoc)

# Para enriquecer la documentación, añade:
# - description a los campos Pydantic (ya lo hicimos en C6)
# - tags a las rutas
# - docstrings a las funciones de ruta

# Ejemplo de cómo FastAPI usa los docstrings como descripción del endpoint:
from fastapi import FastAPI
from pydantic import BaseModel

app_demo = FastAPI(title="Demo docs")

class EjemploEntrada(BaseModel):
    """Datos de entrada de ejemplo."""
    valor: float

@app_demo.post("/ejemplo", tags=["Predicción"], summary="Endpoint de ejemplo")
def ejemplo(datos: EjemploEntrada) -> dict:
    """
    Endpoint de demostración.

    - **valor**: número entre 0 y 100
    - Devuelve el valor duplicado
    """
    return {"resultado": datos.valor * 2}

# Extraer el esquema OpenAPI sin lanzar el servidor
import json
schema = app_demo.openapi()
print("Esquema OpenAPI generado automáticamente:")
print(json.dumps(schema['paths']['/ejemplo'], indent=2))

In [ ]:
# Exportar el esquema OpenAPI del proyecto real a un archivo
import subprocess, json, time
from pathlib import Path

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--port', '8001', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)

import requests
r = requests.get('http://localhost:8001/openapi.json')
if r.status_code == 200:
    schema_completo = r.json()
    Path('docs').mkdir(exist_ok=True)
    with open('docs/openapi.json', 'w', encoding='utf-8') as f:
        json.dump(schema_completo, f, indent=2, ensure_ascii=False)
    print("✓ docs/openapi.json generado")
    print(f"  Endpoints: {list(schema_completo['paths'].keys())}")

proc.terminate(); proc.wait()

---
## 3. README profesional

In [ ]:
%%writefile docs/README.md
# Wine Quality AI System

Sistema de predicción de calidad de vinos que combina Machine Learning y Deep Learning,
expuesto a través de una API REST con FastAPI.

## Arquitectura

```
data/raw/          → Dataset UCI Wine Quality (descarga + SHA256)
data/processed/    → Dataset limpio y con features derivadas
data/db/           → SQLite con historial de predicciones
models/            → Modelos entrenados (.joblib, .keras)
api/               → FastAPI application
docs/              → Documentación (README, OpenAPI)
logs/              → Logs de entrenamiento y API
```

## Instalación rápida

```bash
# Clonar y configurar entorno
git clone https://github.com/TU_USUARIO/wine-quality-ai
cd wine-quality-ai
python -m venv .venv
source .venv/bin/activate   # Linux/Mac
# .venv\Scripts\activate    # Windows

pip install -r requirements.txt
```

## Pipeline completo

```bash
# 1. Descargar datos
jupyter nbconvert --to notebook --execute M9_C1_Obtencion_Datos.ipynb

# 2. Limpiar y preprocesar
jupyter nbconvert --to notebook --execute M9_C2_Limpieza_Preparacion.ipynb

# 3. Almacenar en SQL
jupyter nbconvert --to notebook --execute M9_C3_Almacenamiento_SQL.ipynb

# 4. Entrenar modelos ML
jupyter nbconvert --to notebook --execute M9_C4_Entrenamiento_ML.ipynb

# 5. Entrenar modelos DL
jupyter nbconvert --to notebook --execute M9_C5_Entrenamiento_DL.ipynb

# 6. Lanzar API
uvicorn api.main:app --reload --port 8000
```

## API

| Método | Ruta | Descripción |
|--------|------|-------------|
| GET    | `/health` | Estado del servidor |
| GET    | `/model/info` | Métricas del modelo activo |
| POST   | `/predict` | Predicción individual (RF) |
| POST   | `/predict/dl` | Predicción individual (DL) |
| POST   | `/predict/batch` | Predicción en lote (máx. 100) |

Documentación interactiva: http://localhost:8000/docs

## Ejemplo de uso

```python
import requests

resultado = requests.post('http://localhost:8000/predict', json={
    "tipo": "tinto",
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.7,
    "citric_acid": 0.0,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "pH": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4
}).json()

print(resultado)
# {'prediccion': 'normal', 'probabilidad': 0.32, 'modelo': 'rf_v1', ...}
```

## Rendimiento de los modelos

| Modelo | F1 | AUC | Comentario |
|--------|-----|-----|------------|
| Random Forest (RF) | ~0.73 | ~0.88 | Más rápido en inferencia |
| Red Neuronal (DL)  | ~0.71 | ~0.87 | Estima puntuación exacta |

## Dataset

- **Fuente**: UCI Machine Learning Repository — Wine Quality
- **Tamaño**: 6,497 vinos (1,599 tintos + 4,898 blancos)
- **Target**: clasificación binaria (`buen_vino`: quality ≥ 7)

## Licencia

MIT © 2024

---
## 4. Git y GitHub — flujo básico

In [ ]:
# Git: sistema de control de versiones
# GitHub: plataforma para alojar repositorios Git en la nube

# Flujo de trabajo en un proyecto de ML:
git_commands = """
# 1. Inicializar el repositorio
git init
git remote add origin https://github.com/TU_USUARIO/wine-quality-ai.git

# 2. Crear .gitignore para excluir archivos grandes y secretos
cat .gitignore:
  .venv/
  __pycache__/
  *.pyc
  data/raw/*.csv      # datos crudos (pesados)
  data/raw/*.parquet
  models/*.keras      # modelos (pesados)
  models/*.joblib
  data/db/*.db        # base de datos
  logs/
  .env                # secretos

# 3. Primer commit
git add api/ docs/ M9_*.ipynb requirements.txt .gitignore
git commit -m "feat: pipeline completo M9 — datos, modelos, API"
git push -u origin main

# 4. Flujo de cambios (feature branch)
git checkout -b feature/endpoint-dl
# ... editar api/main.py ...
git add api/main.py
git commit -m "feat: añadir endpoint POST /predict/dl"
git checkout main
git merge feature/endpoint-dl

# 5. Ver historial
git log --oneline --graph
"""

print(git_commands)

In [ ]:
gitignore_content = """# Entornos virtuales
.venv/
env/
venv/

# Python
__pycache__/
*.pyc
*.pyo
*.egg-info/

# Jupyter
.ipynb_checkpoints/

# Datos (no subir a GitHub — usar DVC o descargar con el script)
data/raw/*.csv
data/raw/*.parquet
data/db/

# Modelos entrenados (pueden ser muy grandes)
models/*.keras
models/*.joblib
models/*.pkl

# Logs
logs/

# Secretos
.env
*.key

# macOS
.DS_Store
"""

with open('.gitignore', 'w') as f:
    f.write(gitignore_content)
print(".gitignore creado")

---
## 5. `requirements.txt` y reproducibilidad

In [ ]:
# Generar requirements.txt con las versiones exactas
import subprocess
result = subprocess.run(['pip', 'freeze'], capture_output=True, text=True)

# Filtrar solo los paquetes que usamos en el proyecto
paquetes_proyecto = [
    'pandas', 'numpy', 'scikit-learn', 'tensorflow', 'fastapi',
    'uvicorn', 'pydantic', 'requests', 'sqlalchemy', 'joblib',
    'matplotlib', 'seaborn', 'scipy', 'pyarrow'
]

lineas_filtradas = []
for linea in result.stdout.splitlines():
    pkg = linea.split('==')[0].lower()
    if any(p in pkg for p in paquetes_proyecto):
        lineas_filtradas.append(linea)

requirements = '\n'.join(sorted(lineas_filtradas))
with open('requirements.txt', 'w') as f:
    f.write(requirements + '\n')

print("requirements.txt generado:")
print(requirements)

In [ ]:
# Listar la estructura del proyecto
from pathlib import Path

def listar_estructura(raiz='.', excluir={'.git', '__pycache__', '.venv', '.ipynb_checkpoints'}):
    for p in sorted(Path(raiz).rglob('*')):
        if any(exc in p.parts for exc in excluir):
            continue
        nivel = len(p.relative_to(raiz).parts) - 1
        sangria = '  ' * nivel
        if p.is_dir():
            print(f"{sangria}📁 {p.name}/")
        else:
            size = p.stat().st_size
            size_str = f"{size/1024:.0f}KB" if size > 1024 else f"{size}B"
            print(f"{sangria}  {p.name} ({size_str})")

print("Estructura del proyecto:")
listar_estructura()

---
## Práctica

### Ejercicio 1 — Docstrings para la API

**Enunciado:** Añade docstrings completos (Google-style) a las dos funciones más importantes de `api/main.py`:
1. `entrada_a_array(vino: VinoEntrada) -> np.ndarray`
2. `predecir(vino: VinoEntrada) -> PrediccionSalida`

Incluye: descripción, Args, Returns, Raises y Example.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
# Edita api/main.py y reemplaza las funciones con estas versiones documentadas:

def entrada_a_array(vino: VinoEntrada) -> np.ndarray:
    """
    Convierte el modelo Pydantic de entrada al array NumPy que espera el modelo.

    Calcula tres features derivadas antes de construir el array:
    - so2_ratio: proporción de SO2 libre respecto al total
    - acid_index: diferencia entre acidez fija y volátil
    - density_diff: distancia de la densidad al agua pura (1.0 g/cm³)

    Args:
        vino: Instancia validada de VinoEntrada con los valores fisicoquímicos.

    Returns:
        Array de forma (1, 15) con las 15 features en el orden esperado por el modelo.

    Example:
        >>> v = VinoEntrada(tipo='tinto', fixed_acidity=7.4, ...)
        >>> X = entrada_a_array(v)
        >>> X.shape
        (1, 15)
    """
    d = {
        'fixed acidity': vino.fixed_acidity,
        # ... resto de campos ...
    }
    d['so2_ratio']    = d['free sulfur dioxide'] / (d['total sulfur dioxide'] + 1e-8)
    d['acid_index']   = d['fixed acidity'] - d['volatile acidity']
    d['density_diff'] = abs(d['density'] - 1.0)
    d['tipo_encoded'] = 1 if vino.tipo == 'tinto' else 0
    return np.array([[d[f] for f in FEATURES_ORDEN]])

@app.post("/predict", response_model=PrediccionSalida, tags=["Predicción"])
def predecir(vino: VinoEntrada) -> PrediccionSalida:
    """
    Predice si un vino es de buena calidad.

    Usa el modelo Random Forest entrenado en M9_C4 para clasificar el vino
    como 'bueno' (quality ≥ 7) o 'normal' (quality < 7).

    Args:
        vino: Características fisicoquímicas del vino a evaluar.

    Returns:
        PrediccionSalida con la clasificación, probabilidad y metadata.

    Raises:
        HTTPException (500): Si el archivo del modelo no está disponible.
        HTTPException (422): Si los datos de entrada no pasan la validación Pydantic.
    """
    modelo = cargar_modelo_rf()
    X = entrada_a_array(vino)
    pred = int(modelo.predict(X)[0])
    prob = float(modelo.predict_proba(X)[0, 1])
    return PrediccionSalida(
        prediccion='bueno' if pred else 'normal',
        probabilidad=round(prob, 4),
        modelo='rf_v1',
        timestamp=datetime.now().isoformat()
    )
```
</details>

---
### Ejercicio 2 — Generar documentación con `pdoc`

**Enunciado:** `pdoc` convierte docstrings en HTML automáticamente:

1. `pip install pdoc`
2. Crea un módulo `src/preprocessing.py` con la función `preprocesar_vinos` documentada
3. `pdoc src/preprocessing.py -o docs/api_docs/`
4. Abre el HTML generado y verifica el resultado

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from pathlib import Path
import subprocess

# Crear el módulo
Path('src').mkdir(exist_ok=True)

codigo = '''
"""
Módulo de preprocesado del dataset Wine Quality.

Contiene funciones para limpiar, transformar y enriquecer
el dataset de UCI Wine Quality antes del entrenamiento.
"""
import pandas as pd
import numpy as np

def preprocesar_vinos(
    df: pd.DataFrame,
    umbral_calidad: int = 7,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Limpia y enriquece el dataset de calidad de vinos.

    Args:
        df: DataFrame con columnas del dataset Wine Quality.
        umbral_calidad: Puntuación mínima para clasificar como bueno.
        verbose: Si True, imprime el shape del resultado.

    Returns:
        DataFrame procesado con 'so2_ratio' y 'buen_vino' añadidos.

    Example:
        >>> df_proc = preprocesar_vinos(df_raw, umbral_calidad=7)
    """
    df_proc = df.dropna().drop_duplicates().copy()
    df_proc['so2_ratio'] = df_proc['free sulfur dioxide'] / (df_proc['total sulfur dioxide'] + 1e-8)
    df_proc['buen_vino'] = (df_proc['quality'] >= umbral_calidad).astype(int)
    if verbose:
        print(f"{df_proc.shape}")
    return df_proc
'''
Path('src/preprocessing.py').write_text(codigo)

# Instalar pdoc si es necesario
subprocess.run(['pip', 'install', 'pdoc', '-q'])

# Generar HTML
Path('docs/api_docs').mkdir(parents=True, exist_ok=True)
result = subprocess.run(['pdoc', 'src/preprocessing.py', '-o', 'docs/api_docs/'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Documentación generada en docs/api_docs/")
else:
    print(result.stderr)
```
</details>

---
### Ejercicio 3 — CHANGELOG.md

**Enunciado:** Crea un `docs/CHANGELOG.md` siguiendo el estándar [Keep a Changelog](https://keepachangelog.com):

```markdown
## [Unreleased]
## [1.0.0] - 2024-XX-XX
### Added
### Changed
### Fixed
```

Documenta las 8 fases del proyecto con sus principales decisiones técnicas.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
## Resumen

```python
# Docstring Google-style
def mi_funcion(x: int, y: float = 1.0) -> dict:
    """
    Descripción en una línea.

    Explicación adicional si es necesaria.

    Args:
        x: Descripción del parámetro x.
        y: Descripción del parámetro y.

    Returns:
        Descripción del valor devuelto.

    Raises:
        ValueError: Cuándo se lanza.
    """
    ...

# Git básico
# git init
# git add <archivos>
# git commit -m "tipo: descripción concisa"
# git push origin main
```

## ¿Qué viene en la próxima clase?
En la **Clase 8** desplegaremos el sistema en local: servicio `systemd` en Linux y contenedor **Docker**.